# MEIKA Vox · transcripción desde Google Drive

**Un solo paso:** pulsa **Ejecutar** (o **Ejecutar todo**). MEIKA Vox prepara el motor, verifica las dependencias, conecta Drive y abre el panel.
Si Drive falla, repite la misma celda: en esta sesión no se reinstalará el motor ya comprobado.

**Seguridad:** Google Colab puede pedirte confirmar la ejecución de un cuaderno de GitHub y autorizar tu Google Drive. Son permisos de Google que la herramienta no puede omitir. Comprueba que la URL corresponde al repositorio MEIKA Vox antes de conceder acceso.

Con GPU disponible usa WhisperX `large-v3`; sin GPU puedes seleccionar CPU con modelo `small`. Los resultados deben revisarse antes de citarlos.


In [ ]:
#@title MEIKA Vox · Preparar y abrir automáticamente
import importlib
import importlib.metadata as metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import tomllib
from pathlib import Path

from google.colab import drive, output

if any(t.name in {"MEIKA_Vox_batch", "MEIKA_Vox_search"} and t.is_alive()
       for t in threading.enumerate()):
    raise RuntimeError("Hay un lote activo. Espera a que termine.")

REPO = Path("/content/meika-vox")
REPO_REF = "main"
REPORT = Path("/content/meika_vox_preparacion.txt")
_started = time.perf_counter()
_timings = []
_state = "iniciando"

def report():
    rows = ["MEIKA Vox · preparación",
            f"Python: {sys.version.split()[0]}",
            f"Repositorio: {REPO_REF}",
            f"Estado: {_state}"]
    rows.extend(f"{label}: {elapsed:.2f} s" for label, elapsed in _timings)
    rows.append(f"Total: {time.perf_counter() - _started:.2f} s")
    REPORT.write_text("\n".join(rows) + "\n", encoding="utf-8")

def timed(label, action):
    global _state
    _state = label
    report()
    print(f"Preparando: {label}…", flush=True)
    t = time.perf_counter()
    try:
        return action()
    finally:
        elapsed = time.perf_counter() - t
        _timings.append((label, elapsed))
        print(f"{label}: {elapsed:.1f} s", flush=True)
        report()

def checked(command, *, live=False):
    args = [str(x) for x in command]
    if live:
        result = subprocess.run(args, check=False)
    else:
        result = subprocess.run(args, capture_output=True, text=True, check=False)
    if result.returncode:
        if not live:
            print(result.stderr[-6000:] or result.stdout[-6000:])
        raise RuntimeError("Falló la preparación; revisa el error anterior.")

def prepare_tools():
    missing = [name for name in ("ffmpeg", "ffprobe", "git") if not shutil.which(name)]
    if not missing:
        print("FFmpeg, ffprobe y Git ya están disponibles.")
        return
    checked(["apt-get", "-qq", "update"])
    checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"])

def update_code():
    if not (REPO / ".git").exists():
        checked(["git", "clone", "--quiet", "https://github.com/MeikaLab/meika-vox.git", REPO])
    checked(["git", "-C", REPO, "fetch", "--quiet", "origin", REPO_REF])
    checked(["git", "-C", REPO, "checkout", "--quiet", "--detach", "FETCH_HEAD"])

def prepare_packages():
    # Use the project's version pins: never bypass compatibility with --no-deps.
    project = tomllib.loads((REPO / "pyproject.toml").read_text(encoding="utf-8"))
    packages = project["project"]["optional-dependencies"]["whisperx"]
    exact = {}
    for item in packages:
        if "==" in item:
            name, version = item.split("==", 1)
            exact[name] = version
    missing = []
    for name, version in exact.items():
        try:
            installed = metadata.version(name).split("+")[0]
        except metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            missing.append(f"{name}: {installed or 'sin instalar'} → {version}")
    try:
        metadata.version("meika-vox")
        metadata.version("ipywidgets")
    except metadata.PackageNotFoundError:
        missing.append("MEIKA Vox / controles del panel")
    if not missing:
        print("Dependencias compatibles: instalación omitida.")
        return
    print("Instalando versiones necesarias:", "; ".join(missing), flush=True)
    loaded = set(sys.modules).intersection({
        "torch", "torchaudio", "torchvision", "transformers", "numpy"
    })
    if loaded:
        print("Colab ya había cargado bibliotecas: " + ", ".join(sorted(loaded)))
        print("Se instalarán las versiones requeridas. WhisperX se ejecutará en")
        print("un proceso Python independiente para evitar mezclas en el kernel.")
        # Do not reload already imported torch/numpy/transformers inside IPython.

    checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "-e", str(REPO) + "[whisperx]", "ipywidgets>=8,<9"], live=True)

def check_engine():
    # A fresh interpreter catches the real transformers.Pipeline import failure.
    global probe
    probe = subprocess.run([sys.executable, "-c", """
import torch, transformers, whisperx
import whisperx.asr, whisperx.alignment
from transformers import Pipeline
print("WhisperX listo:", whisperx.__file__)
print("MEIKA_CUDA_AVAILABLE=" + ("1" if torch.cuda.is_available() else "0"))
"""], capture_output=True, text=True, timeout=180)
    if probe.returncode:
        print(probe.stderr[-12000:] or probe.stdout[-12000:])
        raise RuntimeError("El motor no pudo importarse. Revisa el error anterior.")
    for name in list(sys.modules):
        if name == "meika_vox" or name.startswith("meika_vox."):
            del sys.modules[name]
    if str(REPO / "src") not in sys.path:
        sys.path.insert(0, str(REPO / "src"))
    # GPU is determined by a fresh interpreter, not Colab's potentially stale torch.
    gpu_lines = [line for line in probe.stdout.splitlines()
                 if line.startswith("MEIKA_CUDA_AVAILABLE=")]
    if len(gpu_lines) != 1:
        raise RuntimeError("No se pudo comprobar el estado de la GPU en el proceso limpio.")
    os.environ["MEIKA_VOX_CUDA_AVAILABLE"] = gpu_lines[0].split("=", 1)[1]
    print("Importaciones y versiones comprobadas en un intérprete independiente.")

# Mark readiness only after a clean independent interpreter imported the engine.
# Retained across retries inside this VM, not across Colab's ephemeral sessions.
READY_MARKER = Path("/content/meika_vox_preparado.json")

def cached_ready():
    try:
        ready = json.loads(READY_MARKER.read_text(encoding="utf-8"))
        if ready["python"] != sys.version.split()[0] or ready["repo"] != REPO_REF:
            return False
        if not (REPO / ".git").exists():
            return False
        commit = subprocess.run(
            ["git", "-C", str(REPO), "rev-parse", "HEAD"],
            capture_output=True, text=True, check=False,
        )
        if commit.returncode or commit.stdout.strip() != ready["commit"]:
            return False
        installed = tomllib.loads((REPO / "pyproject.toml").read_text(encoding="utf-8"))
        for pin in installed["project"]["optional-dependencies"]["whisperx"]:
            if "==" in pin:
                name, version = pin.split("==", 1)
                if metadata.version(name).split("+")[0] != version:
                    return False
        if metadata.version("ipywidgets").split(".")[0] != "8":
            return False
        os.environ["MEIKA_VOX_CUDA_AVAILABLE"] = str(ready["cuda"])
        if str(REPO / "src") not in sys.path:
            sys.path.insert(0, str(REPO / "src"))
        return True
    except (OSError, KeyError, ValueError, metadata.PackageNotFoundError):
        return False

try:
    if cached_ready():
        _state = "preparado (reutilizado)"
        print("Motor ya preparado: no se reinstala al intentar conectar Drive.")
    else:
        READY_MARKER.unlink(missing_ok=True)
        timed("FFmpeg y Git", prepare_tools)
        timed("Repositorio", update_code)
        timed("Dependencias", prepare_packages)
        timed("Comprobación real del motor", check_engine)
        commit = subprocess.run(
            ["git", "-C", str(REPO), "rev-parse", "HEAD"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        READY_MARKER.write_text(json.dumps({
            "python": sys.version.split()[0], "repo": REPO_REF, "commit": commit,
            "cuda": os.environ["MEIKA_VOX_CUDA_AVAILABLE"],
        }), encoding="utf-8")
        _state = "preparado"
except Exception:
    _state = "falló"
    report()
    print("Informe:", REPORT)
    raise
report()
print(f"Motor disponible en {time.perf_counter() - _started:.1f} s")



# Confirmación de Colab y autorización de Drive son acciones de Google.
REPO = Path("/content/meika-vox")
if not (REPO / "src/meika_vox").is_dir():
    raise RuntimeError("Primero ejecuta «1 · Preparar MEIKA Vox».")
if any(t.name in {"MEIKA_Vox_batch", "MEIKA_Vox_search"} and t.is_alive()
       for t in threading.enumerate()):
    raise RuntimeError("Hay una transcripción en curso; espera a que termine.")

start_drive = time.perf_counter()
print("Conectando Google Drive…", flush=True)
try:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    else:
        print("Google Drive ya conectado: no se solicita autorización de nuevo.")
except (ValueError, OSError) as exc:
    print("Google Drive no se conectó. El motor ya quedó preparado.")
    print("Comprueba la autorización en tu cuenta y vuelve a pulsar Ejecutar.")
    print("En esta sesión no volverá a instalarse WhisperX.")
    raise RuntimeError("Fallo de autorización/montaje de Google Drive.") from exc

root = Path("/content/drive/MyDrive")
if not root.is_dir():
    raise RuntimeError("Drive se montó, pero no aparece Mi unidad. Revisa la cuenta autorizada.")
print(f"Drive conectado en {time.perf_counter() - start_drive:.1f} s")
output.enable_custom_widget_manager()
spec = importlib.util.spec_from_file_location("vox_ui", REPO / "scripts/colab_ui.py")
ui = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ui)
print(f"Drive verificado; preparando panel: {time.perf_counter() - start_drive:.1f} s")
_panel = ui.build_panel(REPO, root)
